# 第 18 章 · DQN：神经网络 + 经验回放 + 目标网络

本章回答：把 Q-learning 的表格换成神经网络，会遇到什么麻烦？DQN 用了哪两个招数让训练稳下来？loss 到底在算什么？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/18-dqn.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np
import random
import torch
from torch import nn

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 手算 y | `td_target(0.0, torch.tensor([0.5, 0.8]), False)` | 0.72 |
| 回放缓冲区 | `buffer.append(...)`；`buffer[-2000:]`；`random.sample(buffer, 32)` | — |
| 目标网络 | `target.load_state_dict(q.state_dict())`（每 100 步） | — |
| 训练 3000 步 | `train_dqn()` | Q(·, 右) = 0.656、0.729、0.81、0.9、1.0 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `CORRIDOR_N`**（来自 `rl_envs.py`）

In [ ]:
CORRIDOR_N = 6

**常量 `GAMMA`**（来自 `ch18_dqn.py`）

In [ ]:
GAMMA = 0.9

**函数 `one_hot`**（来自 `ch18_dqn.py`）：下面要用到的小工具。

In [ ]:
def one_hot(s):
    x = torch.zeros(CORRIDOR_N)
    x[s] = 1.0
    return x

**函数 `td_target`**（来自 `ch18_dqn.py`）：下面要用到的小工具。

In [ ]:
def td_target(r, q_next_target, done, gamma=GAMMA):
    return r + (0.0 if done else gamma * q_next_target.max().item())

**函数 `corridor_step`**（来自 `rl_envs.py`）：下面要用到的小工具。

In [ ]:
def corridor_step(s, a):
    ns = max(s - 1, 0) if a == 0 else s + 1
    if ns == CORRIDOR_N - 1:
        return ns, 1.0, True
    return ns, 0.0, False

**函数 `train_dqn`**（来自 `ch18_dqn.py`）：下面要用到的小工具。

In [ ]:
def train_dqn(steps=3000, seed=0, eps=0.3, batch=32, sync_every=100, lr=0.01):
    torch.manual_seed(seed)
    random.seed(seed)
    q = nn.Sequential(nn.Linear(CORRIDOR_N, 32), nn.ReLU(), nn.Linear(32, 2))
    target = nn.Sequential(nn.Linear(CORRIDOR_N, 32), nn.ReLU(), nn.Linear(32, 2))
    target.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=lr)
    buffer, s = [], 0
    for t in range(steps):
        with torch.no_grad():
            a = random.randrange(2) if random.random() < eps else int(q(one_hot(s)).argmax())
        ns, r, done = corridor_step(s, a)
        buffer.append((s, a, r, ns, done))
        buffer = buffer[-2000:]  # 经验回放：存最近 2000 条
        s = 0 if done else ns
        if len(buffer) >= batch:
            b = random.sample(buffer, batch)  # 随机抽，打乱前后相关
            S = torch.stack([one_hot(x[0]) for x in b])
            A = torch.tensor([x[1] for x in b])
            R = torch.tensor([x[2] for x in b])
            NS = torch.stack([one_hot(x[3]) for x in b])
            D = torch.tensor([float(x[4]) for x in b])
            with torch.no_grad():
                y = R + GAMMA * (1 - D) * target(NS).max(dim=1).values  # 目标网络算 y，不求梯度
            pred = q(S).gather(1, A[:, None]).squeeze(1)
            loss = ((y - pred) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        if t % sync_every == 0:
            target.load_state_dict(q.state_dict())  # 隔一阵子才把目标网络对齐
    return q

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("手算 y：r = 0，Q_目标(s') = [0.5, 0.8]，y =", td_target(0.0, torch.tensor([0.5, 0.8]), False))

In [ ]:
q = train_dqn()

In [ ]:
with torch.no_grad():
        Q = torch.stack([q(one_hot(s)) for s in range(CORRIDOR_N - 1)])

In [ ]:
print("学到的 Q（每行 [左, 右]）：\n", Q.numpy().round(3))

In [ ]:
print("理论 Q(s, 右) = γ^(4−s)：", [round(GAMMA ** (4 - s), 3) for s in range(CORRIDOR_N - 1)])

## 3. 代码里才看得到的

1. **`torch.no_grad()`**：告诉 PyTorch 这一段不要记录梯度：目标 y 当成常数（第三步"目标固定"在代码里就是这一行）；
2. **`gather`**：网络一次输出两个动作的 Q，但 loss 只能用"实际做了的那个动作"的 Q，`gather(1, A)` 按动作编号把它挑出来；
3. **`(1 - D)`**：回合结束时没有下一步，乘 0 把 $\gamma\max Q$ 去掉；忘了这一项是 DQN 最常见的 bug 之一。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. `train_dqn(sync_every=1000)`：目标网络 1000 步才对齐一次，3000 步后 Q(·, 右) 是 0.09、0.055、0.81、0.9、1.0，离终点远的两格还没学到。目标网络对齐得太慢，价值往回传也慢。
2. `train_dqn(steps=500)` → 0.66、0.73、0.808、0.902、1.001：这个小走廊 500 步就够了。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> Q 表换网络，输出每个动作的 Q；y 当答案做回归，loss = (y − Q)²；
> 数据前后太像，存进盒子随机抽（经验回放）；能用旧数据靠 off-policy；
> 目标里有 θ 会晃，复制一份 θ⁻ 隔一阵对齐（目标网络）；
> 采样、存、抽、算 y、下降、对齐，就是 DQN。